In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import sys
sys.path.append("../")
from src.preprocessing.pdf_extractor import PDFExtractor
from src.preprocessing.document_validator import DocumentValidator

In [3]:
# ---------------------------------------------------------
# LOAD FILES
# ---------------------------------------------------------

VOCAB_FILE = "../data/vocabulary/ministry_tfidf_vocab.json"
EMBED_FILE = "../data/embeddings/ministry_embeddings.json"


def load_vocab():
    with open(VOCAB_FILE, "r") as f:
        return json.load(f)


def load_embeddings():
    with open(EMBED_FILE, "r") as f:
        data = json.load(f)

    # convert to numpy
    for k in data:
        data[k] = np.array(data[k])

    return data

In [4]:
# ---------------------------------------------------------
# EXTRACT PARAGRAPHS FROM PDF
# ---------------------------------------------------------

def extract_paragraphs(pdf_path):

    extractor = PDFExtractor()
    result = extractor.extract(pdf_path)

    paragraphs = [seg["paragraph"] for seg in result["segments"]]

    return paragraphs

In [10]:
# ---------------------------------------------------------
# VALIDATE DOCUMENT
# ---------------------------------------------------------

def validate(text):

    validator = DocumentValidator(vocab_path="../data/vocabulary/parliament_vocab.txt")
    results = validator.validate_document(text)

    return results

In [6]:
# ---------------------------------------------------------
# VOCAB BASED CLASSIFICATION
# ---------------------------------------------------------

def classify_vocab(paragraph, ministry_vocab):

    words = paragraph.lower().split()

    scores = {}

    for ministry, vocab in ministry_vocab.items():

        count = 0

        for term in vocab:
            if term.lower() in paragraph.lower():
                count += 1

        scores[ministry] = count

    total = sum(scores.values()) + 1e-9

    probs = {k: v / total for k, v in scores.items()}

    return probs


# ---------------------------------------------------------
# EMBEDDING BASED CLASSIFICATION
# ---------------------------------------------------------

def classify_embedding(paragraph, model, ministry_embeddings):

    p_emb = model.encode(paragraph)

    scores = {}

    for ministry, m_emb in ministry_embeddings.items():

        sim = cosine_similarity(
            p_emb.reshape(1, -1),
            m_emb.reshape(1, -1)
        )[0][0]

        scores[ministry] = float(sim)

    # softmax normalization
    exp_scores = np.exp(list(scores.values()))
    probs = exp_scores / np.sum(exp_scores)

    return dict(zip(scores.keys(), probs))

In [7]:
# ---------------------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------------------

def run_pipeline(pdf_path):

    print(f"Processing: {pdf_path}")
    print("-" * 50)

    paragraphs = extract_paragraphs(pdf_path)

    full_text = "\n\n".join(paragraphs)

    validation = validate(full_text)

    print("VALIDATION RESULTS:")
    print(json.dumps(validation, indent=4))

    if not validation["is_valid"]:
        print("Document failed validation")
        return

    print("\nLoading ministry vocab...")
    ministry_vocab = load_vocab()

    print("Loading ministry embeddings...")
    ministry_embeddings = load_embeddings()

    model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder="../models")

    print("\nClassifying paragraphs...\n")

    results = []

    for i, para in enumerate(paragraphs):

        vocab_probs = classify_vocab(para, ministry_vocab)

        embed_probs = classify_embedding(
            para,
            model,
            ministry_embeddings
        )

        results.append({
            "paragraph_id": i,
            "paragraph": para[:200],
            "vocab_distribution": vocab_probs,
            "embedding_distribution": embed_probs
        })

        print("=" * 80)
        print(f"Paragraph {i}")
        # print(para[:200])

        print("\nTop vocab ministry:")
        print(sorted(vocab_probs.items(), key=lambda x: x[1], reverse=True)[:3])

        print("\nTop embedding ministry:")
        print(sorted(embed_probs.items(), key=lambda x: x[1], reverse=True)[:3])

    return results

In [11]:
# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------

# if __name__ == "__main__":

file_name = "test.pdf"
file_path = f"../data/uploads/{file_name}"

results = run_pipeline(file_path)

Processing: ../data/uploads/test.pdf
--------------------------------------------------
VALIDATION RESULTS:
{
    "is_valid": true,
    "is_english": true,
    "validation_errors": [],
    "vocab_size": 116,
    "required_vocab_matches": 5,
    "parliament_confidence": 0.09482758620689655,
    "matched_vocab_terms": [
        "act",
        "bill",
        "budget",
        "finance bill",
        "gazette",
        "government",
        "half",
        "motion",
        "motion of thanks",
        "schedule",
        "speaker"
    ]
}

Loading ministry vocab...
Loading ministry embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Classifying paragraphs...

Paragraph 0

Top vocab ministry:
[('Ministry_of_Parliamentary_Affairs_MPA_', 0.49999999991666666), ('Ministry_of_Law_and_Justice_MoLJ_', 0.16666666663888888), ('Ministry_of_Minority_Affairs_MoMA_', 0.16666666663888888)]

Top embedding ministry:
[('Ministry_of_Home_Affairs_MHA_', np.float64(0.021151059906757398)), ('Ministry_of_Parliamentary_Affairs_MPA_', np.float64(0.021085025986886965)), ('Ministry_of_Labour_and_Employment_MoLE_', np.float64(0.02078251271932788))]
Paragraph 1

Top vocab ministry:
[('Ministry_of_Parliamentary_Affairs_MPA_', 0.33333333327777775), ('Ministry_of_Textiles_MoT_', 0.16666666663888888), ('Ministry_of_Development_of_North_Eastern_Region_MDoNER_', 0.16666666663888888)]

Top embedding ministry:
[('Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_', np.float64(0.02075773015031245)), ('Ministry_of_Science_and_Technology_MST_', np.float64(0.02059503624578871)), ('Ministry_of_Heavy_Industries_MoHI_', np.float64(0.0205260781736